In [64]:
import pandas as pd

# Load the blood cell anomaly detection dataset
df = pd.read_csv('data/dataset/blood_cell_anomaly_detection.csv')

# Display basic information about the dataset
print("Dataset shape:", df.shape)
print("Columns:", list(df.columns))
print("\nFirst 5 rows:")
df.head()

Dataset shape: (5880, 36)
Columns: ['cell_id', 'cell_type', 'anomaly_label', 'disease_category', 'cell_diameter_um', 'nucleus_area_pct', 'chromatin_density', 'cytoplasm_ratio', 'circularity', 'eccentricity', 'granularity_score', 'lobularity_score', 'membrane_smoothness', 'cell_area_px', 'perimeter_px', 'mean_r', 'mean_g', 'mean_b', 'stain_intensity', 'patient_age_group', 'patient_sex', 'wbc_count_per_ul', 'rbc_count_millions_per_ul', 'hemoglobin_g_dl', 'hematocrit_pct', 'platelet_count_per_ul', 'mcv_fl', 'mchc_g_dl', 'dataset_source', 'staining_protocol', 'microscope_model', 'magnification_x', 'image_resolution_px', 'cytodiffusion_anomaly_score', 'cytodiffusion_classification_confidence', 'labeller_confidence_score']

First 5 rows:


,cell_id,cell_type,anomaly_label,disease_category,cell_diameter_um,nucleus_area_pct,chromatin_density,cytoplasm_ratio,circularity,eccentricity,...,mcv_fl,mchc_g_dl,dataset_source,staining_protocol,microscope_model,magnification_x,image_resolution_px,cytodiffusion_anomaly_score,cytodiffusion_classification_confidence,labeller_confidence_score
0,CELL_005371,Hypersegmented_Neutrophil,1,Infection,15.18,58.8,0.542,0.301,0.563,0.529,...,85.5,31.4,CytoData,Giemsa,Zeiss_Axio,100,224,0.7649,0.5726,0.5670
1,CELL_005300,Hypersegmented_Neutrophil,1,Infection,16.47,73.6,0.583,0.365,0.859,0.443,...,92.5,35.0,PBC_Dataset,Wright,Zeiss_Axio,100,224,0.8472,0.7150,0.7273
2,CELL_000200,Neutrophil,0,Normal_WBC,13.41,55.5,0.448,0.376,0.781,0.407,...,76.3,33.0,CytoData,Wright,Leica_DM2000,100,512,0.0313,0.9225,0.9623
3,CELL_003269,Normal_RBC,0,Normal_RBC,7.36,0.0,0.000,1.000,0.880,0.167,...,92.3,32.5,CytoData,Wright,Leica_DM2000,100,512,0.1293,0.9180,0.8652
4,CELL_003505,Normal_RBC,0,Normal_RBC,7.53,0.0,0.000,1.000,1.000,0.158,...,83.9,33.4,CytoData,Wright,Olympus_BX51,100,224,0.1418,0.9697,0.8898


In [65]:
# Drop unnecessary columns
columns_to_drop = ['cell_area_px', 'perimeter_px', 'cell_id', 'magnification_x', 'image_resolution_px', 'labeller_confidence_score', 'cytodiffusion_classification_confidence', 'microscope_model', 'stain_intensity', 'mean_r', 'mean_g', 'mean_b', 'dataset_source', 'staining_protocol', 'cytodiffusion_anomaly_score']
df.drop(columns=columns_to_drop, inplace=True)

print("Columns after dropping:", list(df.columns))
print("New shape:", df.shape)

Columns after dropping: ['cell_type', 'anomaly_label', 'disease_category', 'cell_diameter_um', 'nucleus_area_pct', 'chromatin_density', 'cytoplasm_ratio', 'circularity', 'eccentricity', 'granularity_score', 'lobularity_score', 'membrane_smoothness', 'patient_age_group', 'patient_sex', 'wbc_count_per_ul', 'rbc_count_millions_per_ul', 'hemoglobin_g_dl', 'hematocrit_pct', 'platelet_count_per_ul', 'mcv_fl', 'mchc_g_dl']
New shape: (5880, 21)


In [66]:
# Prepare feature matrix X and labels y1 (anomaly) and y2 (cell type)
from pathlib import Path
import numpy as np
import pandas as pd

# y1: anomaly label, y2: cell type
y1 = df['anomaly_label']
y2 = df['cell_type']

# X: all columns except the two label columns
X = df.drop(columns=['anomaly_label', 'cell_type'])

# encode y2 as integer codes for modeling (optional)
y2_encoded = y2.astype('category').cat.codes

# Save processed splits
out_dir = Path('data/processed')
out_dir.mkdir(parents=True, exist_ok=True)
X.to_csv(out_dir / 'X.csv', index=False)
y1.to_csv(out_dir / 'y1.csv', index=False)
pd.Series(y2_encoded, name='cell_type_code').to_csv(out_dir / 'y2_encoded.csv', index=False)
# also save compact numpy archive
np.savez_compressed(out_dir / 'dataset_split.npz', X=X.values, y1=y1.values, y2=y2_encoded.values)
print('Saved X, y1, y2_encoded to', out_dir)

Saved X, y1, y2_encoded to data\processed


In [67]:
from sklearn.model_selection import train_test_split

# Create stratified train/validation/test split to maintain class ratios for both y1 and y2
# Combine y1 and y2 into a stratification key to preserve both class distributions
strat_key = y1.astype(str) + "_" + y2_encoded.astype(str)

# First split: 70% train, 30% temp (validation + test)
X_train, X_temp, y1_train, y1_temp, y2_train, y2_temp, strat_train, strat_temp = train_test_split(
    X, y1, y2_encoded, strat_key, test_size=0.3, random_state=42, stratify=strat_key
)

# Second split: split temp into 50% validation, 50% test (which is 15% each of total)
X_val, X_test, y1_val, y1_test, y2_val, y2_test = train_test_split(
    X_temp, y1_temp, y2_temp, test_size=0.5, random_state=42, stratify=strat_temp
)

print(f"Train split: X_train shape = {X_train.shape}, y1_train shape = {y1_train.shape}")
print(f"Val split:   X_val shape = {X_val.shape}, y1_val shape = {y1_val.shape}")
print(f"Test split:  X_test shape = {X_test.shape}, y1_test shape = {y1_test.shape}")

# Verify class balance across splits
print("\n--- Class Distribution ---")
for split_name, y1_s, y2_s in [
    ('Train', y1_train, y2_train),
    ('Val', y1_val, y2_val),
    ('Test', y1_test, y2_test)
]:
    print(f"\n{split_name}:")
    print(f"  y1 (anomaly) value counts:\n{y1_s.value_counts().sort_index()}")
    print(f"  y2 (cell type) unique: {len(y2_s.unique())} types")

# Save splits
splits_dir = Path('data/splits')
splits_dir.mkdir(parents=True, exist_ok=True)

# Save as CSV
for split, (X_s, y1_s, y2_s) in [
    ('train', (X_train, y1_train, y2_train)),
    ('val', (X_val, y1_val, y2_val)),
    ('test', (X_test, y1_test, y2_test))
]:
    X_s.to_csv(splits_dir / f'X_{split}.csv', index=False)
    y1_s.to_csv(splits_dir / f'y1_{split}.csv', index=False)
    pd.Series(y2_s, name='cell_type_code').to_csv(splits_dir / f'y2_{split}.csv', index=False)

# Save as compressed numpy archive
np.savez_compressed(
    splits_dir / 'train_val_test_split.npz',
    X_train=X_train.values, y1_train=y1_train.values, y2_train=y2_train.values,
    X_val=X_val.values, y1_val=y1_val.values, y2_val=y2_val.values,
    X_test=X_test.values, y1_test=y1_test.values, y2_test=y2_test.values
)

print(f"\nSaved stratified train/val/test splits to {splits_dir}")

Train split: X_train shape = (4116, 19), y1_train shape = (4116,)
Val split:   X_val shape = (882, 19), y1_val shape = (882,)
Test split:  X_test shape = (882, 19), y1_test shape = (882,)

--- Class Distribution ---

Train:
  y1 (anomaly) value counts:
anomaly_label
0    2800
1    1316
Name: count, dtype: int64
  y2 (cell type) unique: 19 types

Val:
  y1 (anomaly) value counts:
anomaly_label
0    601
1    281
Name: count, dtype: int64
  y2 (cell type) unique: 19 types

Test:
  y1 (anomaly) value counts:
anomaly_label
0    599
1    283
Name: count, dtype: int64
  y2 (cell type) unique: 19 types

Saved stratified train/val/test splits to data\splits


In [83]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, recall_score, confusion_matrix, accuracy_score


torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



# Encode any non-numeric columns safely
X_train_enc = pd.get_dummies(X_train)
X_val_enc   = pd.get_dummies(X_val)
X_test_enc  = pd.get_dummies(X_test)

X_val_enc = X_val_enc.reindex(columns=X_train_enc.columns, fill_value=0)
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

X_train_enc = X_train_enc.astype(np.float32)
X_val_enc   = X_val_enc.astype(np.float32)
X_test_enc  = X_test_enc.astype(np.float32)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_enc = scaler.fit_transform(X_train_enc)
X_val_enc   = scaler.transform(X_val_enc)
X_test_enc  = scaler.transform(X_test_enc)

# Labels
y_train = np.asarray(y2_train).astype(np.int64)
y_val   = np.asarray(y2_val).astype(np.int64)
y_test  = np.asarray(y2_test).astype(np.int64)

num_classes = len(np.unique(y_train))
input_dim = X_train_enc.shape[1]

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=device)

class ModelMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.net(x)

model_1 = ModelMLP(input_dim, num_classes).to(device)

class ModelNoDropoutMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.net(x)

model_2 = ModelNoDropoutMLP(input_dim, num_classes).to(device)

class ModelIncreaseDropoutMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.net(x)

model_3 = ModelIncreaseDropoutMLP(input_dim, num_classes).to(device)



In [ ]:


# Training loop
def train_model(model, X_train, y_train, X_val, y_val, class_weights_tensor, epochs=100, batch_size=32, learning_rate=0.001):
    from torch.utils.data import TensorDataset, DataLoader

    train_dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    #criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        train_preds = []
        train_labels = []


        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            train_preds.extend(preds.cpu().numpy())
            train_labels.extend(labels.cpu().numpy())

        avg_train_loss = total_loss / (len(X_train) // batch_size)

        # Validation metrics
        model.eval()
        val_preds = []
        val_labels = []

        with torch.no_grad():
            for i in range(0, len(X_val), batch_size):
                inputs = X_val[i:i+batch_size].to(device)
                labels = y_val[i:i+batch_size].to(device)

                outputs = model(inputs)
                _, preds = torch.max(outputs.data, 1)
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())

        val_f1 = f1_score(val_labels, val_preds, average='weighted')
        val_recall = recall_score(val_labels, val_preds, average='weighted')
        val_conf_matrix = confusion_matrix(val_labels, val_preds)
        val_accuracy = accuracy_score(val_labels, val_preds)

        print(f"Epoch {epoch+1}/{epochs}, "
              f"Train Loss: {avg_train_loss:.4f}, "
              f"Val F1 Score: {val_f1:.4f}, "
              f"Val Recall: {val_recall:.4f}, "
              f"Val Accuracy: {val_accuracy:.4f}")
        print("Confusion Matrix:\n", val_conf_matrix)

# Example usage (model_1 is the one with dropout (0.1), model_2 is without dropout, model_3 has increased dropout (0.3)): MODEL 3 BEST
train_model(model_3, torch.tensor(X_train_enc, dtype=torch.float32),
            torch.tensor(y_train, dtype=torch.long),
            torch.tensor(X_val_enc, dtype=torch.float32),
            torch.tensor(y_val, dtype=torch.long),
            class_weights_tensor)

Epoch 1/100, Train Loss: 0.0005, Val F1 Score: 0.9608, Val Recall: 0.9615, Val Accuracy: 0.9615
Confusion Matrix:
 [[ 12   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0]
 [  0  22   0   0   1   0   0   0   0   0   0   0   0   0   0   0   0   0
    0]
 [  0   0  33   0   0   0   0   0   0   0   0   9   0   0   0   0   0   0
    0]
 [  0   0   0  30   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0]
 [  0   2   0   0  40   0   0   0   3   0   0   0   0   0   0   0   0   0
    0]
 [  0   0   0   0   0  24   0   0   0   0   0   0   0   0   0   0   0   0
    0]
 [  0   0   0   0   0   0 128   0   0   0   0   0   0   0   0   0   0   0
    0]
 [  0   0   0   0   0   0   0  60   0   0   0   0   0   0   0   0   0   0
    0]
 [  0   0   0   0   2   0   0   1 162   0   0   0   0   0   0   0   0   0
    0]
 [  0   0   0   0   0   0   0   0   0 135   0   0   0   0   0   0   0   0
    0]
 [  0   0   0   0   0   0   0   0   0   0  45   0   0   0   0   0   0   0
 

In [72]:
from sklearn.metrics import f1_score, recall_score, confusion_matrix, accuracy_score

def evaluate_model(y_true, y_pred, name="Model"):
    f1 = f1_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    print(f"\n{name} Results:")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Recall: {recall:.4f}")
    print("Confusion Matrix:\n", cm)

    return acc, f1, recall, cm

In [ ]:
from sklearn.linear_model import LogisticRegression

#baseline (BEST)
logreg_1 = LogisticRegression(C=1.0,penalty='l2',solver='lbfgs',max_iter=1000)

#increased regularization
logreg_2 = LogisticRegression(C=2.0,penalty='l2',solver='lbfgs',max_iter=1000)

#decreased regularization
logreg_3 = LogisticRegression(C=0.5,penalty='l2',solver='lbfgs',max_iter=1000)

#TOGGLE here ----
logreg = logreg_1

logreg.fit(X_train_enc, y_train)
logreg_preds = logreg.predict(X_val_enc)

evaluate_model(y_val, logreg_preds, "Logistic Regression")


Logistic Regression Results:
Accuracy: 0.9751
F1 Score: 0.9746
Recall: 0.9751
Confusion Matrix:
 [[ 12   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0]
 [  0  22   0   0   1   0   0   0   0   0   0   0   0   0   0   0   0   0
    0]
 [  0   0  37   0   0   0   0   0   0   0   0   5   0   0   0   0   0   0
    0]
 [  0   0   0  30   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0]
 [  0   2   0   0  42   0   0   0   1   0   0   0   0   0   0   0   0   0
    0]
 [  0   0   0   0   0  24   0   0   0   0   0   0   0   0   0   0   0   0
    0]
 [  0   0   0   0   0   0 128   0   0   0   0   0   0   0   0   0   0   0
    0]
 [  0   0   0   0   0   0   0  60   0   0   0   0   0   0   0   0   0   0
    0]
 [  0   1   0   0   0   0   0   0 164   0   0   0   0   0   0   0   0   0
    0]
 [  0   0   0   0   0   0   0   0   0 135   0   0   0   0   0   0   0   0
    0]
 [  0   0   0   0   0   0   0   0   0   0  45   0   0   0   0   0   0   0
    0]
 [  0   0  

c:\Users\rvazq\miniconda3\envs\csce583_torch\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


(0.9750566893424036,
 0.9745685971876447,
 0.9750566893424036,
 array([[ 12,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0],
        [  0,  22,   0,   0,   1,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0],
        [  0,   0,  37,   0,   0,   0,   0,   0,   0,   0,   0,   5,   0,
           0,   0,   0,   0,   0,   0],
        [  0,   0,   0,  30,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0],
        [  0,   2,   0,   0,  42,   0,   0,   0,   1,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0],
        [  0,   0,   0,   0,   0,  24,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0],
        [  0,   0,   0,   0,   0,   0, 128,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0],
        [  0,   0,   0,   0,   0,   0,   0,  60,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0],
        [  0,   1,   0,  

In [ ]:
from sklearn.ensemble import RandomForestClassifier

#baseline
rf_1 = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)

#no max depth
rf_2 = RandomForestClassifier(n_estimators=200, random_state=42)

#increased max depth (BEST)
rf_3 = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42)

#TOGGLE here ----
rf = rf_3

rf.fit(X_train_enc, y_train)

rf_preds = rf.predict(X_val_enc)

evaluate_model(y_val, rf_preds, "Random Forest")


Random Forest Results:
Accuracy: 0.9773
F1 Score: 0.9764
Recall: 0.9773
Confusion Matrix:
 [[ 12   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0]
 [  0  22   0   0   1   0   0   0   0   0   0   0   0   0   0   0   0   0
    0]
 [  0   0  40   0   0   0   0   0   0   0   0   2   0   0   0   0   0   0
    0]
 [  0   0   0  30   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0]
 [  0   1   0   0  43   0   0   0   1   0   0   0   0   0   0   0   0   0
    0]
 [  0   0   0   0   0  24   0   0   0   0   0   0   0   0   0   0   0   0
    0]
 [  0   0   0   0   0   0 128   0   0   0   0   0   0   0   0   0   0   0
    0]
 [  0   0   0   0   0   0   0  60   0   0   0   0   0   0   0   0   0   0
    0]
 [  0   0   0   0   2   0   0   0 163   0   0   0   0   0   0   0   0   0
    0]
 [  0   0   0   0   0   0   0   0   0 135   0   0   0   0   0   0   0   0
    0]
 [  0   0   0   0   0   0   0   0   0   0  45   0   0   0   0   0   0   0
    0]
 [  0   0  12   0

(0.9773242630385488,
 0.9763914424400055,
 0.9773242630385488,
 array([[ 12,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0],
        [  0,  22,   0,   0,   1,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0],
        [  0,   0,  40,   0,   0,   0,   0,   0,   0,   0,   0,   2,   0,
           0,   0,   0,   0,   0,   0],
        [  0,   0,   0,  30,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0],
        [  0,   1,   0,   0,  43,   0,   0,   0,   1,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0],
        [  0,   0,   0,   0,   0,  24,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0],
        [  0,   0,   0,   0,   0,   0, 128,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0],
        [  0,   0,   0,   0,   0,   0,   0,  60,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0],
        [  0,   0,   0,  